# Climate Change Data Analysis

Author: Tom Rowan, st20285213

## Introduction

Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.

## Import Libraries

In [43]:
import os
import re
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import scipy.cluster.hierarchy as shc
from sklearn.cluster import AgglomerativeClustering
%matplotlib inline

## Task 1 - Import and Preprocess Data Sets

Task 1:
i.	Download the dataset from the above link. 
ii.	Do the necessary preprocessing for example i.e., check for any null value or outlier. If found replace that with mean value and delete repeating factors as necessary.


Reference: https://databank.worldbank.org/reports.aspx?source=2&Topic=19#advancedDownloadOptions

Accessed: 10/05/2024

Reference: https://ourworldindata.org/world-region-map-definitions

Accessed:  15/05/2024



In [72]:
# World Bank Country/Continent Data
countries = pd.read_csv('../datasets/countries/world-regions-according-to-the-world-bank.csv')
regions = countries['World Region according to the World Bank'].unique()
regions.sort()

countries[['Entity','World Region according to the World Bank']].groupby(['World Region according to the World Bank']).count()

# Work Bank Development Data
#wbd = pd.read_csv('../datasets/wbank_dev_indicators/38b7cb7a-172f-4be9-a48d-34d89fca9fc9_Data.csv')
#wbd.head()


,Entity
World Region according to the World Bank,
East Asia and Pacific,37
Europe and Central Asia,58
Latin America and Caribbean,42
Middle East and North Africa,21
North America,3
South Asia,8
Sub-Saharan Africa,48


In [45]:
wbd.loc[wbd['Series Code'] == "AG.LND.AGRI.K2"]

,Series Name,Series Code,Country Name,Country Code,2000 [YR2000],2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],...,2014 [YR2014],2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023]
0,Agricultural land (sq. km),AG.LND.AGRI.K2,Afghanistan,AFG,377940,377950,377900,378840,379280,379170,...,379100,379100,379100,379100,380100,380100,383130,383130,..,..
1,Agricultural land (sq. km),AG.LND.AGRI.K2,Albania,ALB,11440,11390,11400,11210,11220,10770,...,11742.9,11743,11817,11742.81,11740.81,11740,11655.55,11363.3,..,..
2,Agricultural land (sq. km),AG.LND.AGRI.K2,Algeria,DZA,400210,401090,398550,399057,411450,412110,...,414310,414564,413602,413351.408,413388.496,413160.71,413160.71,413160.71,..,..
3,Agricultural land (sq. km),AG.LND.AGRI.K2,American Samoa,ASM,22.7,23,23.3,23.6,23.9,24.2,...,26.9,27.2,27.5,27.8,28.1,28.4,28.7,29,..,..
4,Agricultural land (sq. km),AG.LND.AGRI.K2,Andorra,AND,230,227.5,228.5,228.6,228.1,218,...,188,188.1,188.2,188.2,188.34,188.001,187.773,187.562,..,..
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,Agricultural land (sq. km),AG.LND.AGRI.K2,Sub-Saharan Africa,SSF,9816756.063,9853189.038,9877442.818,9937208.871,9953818.379,10035523.13,...,10391986.8,10420918.54,10444995.23,10468340.43,10500976.83,10518137.82,10532028.68,10542626.28,..,..
262,Agricultural land (sq. km),AG.LND.AGRI.K2,Sub-Saharan Africa (excluding high income),SSA,9816716.063,9853149.038,9877402.818,9937168.871,9953778.379,10035483.13,...,10391971.3,10420903.04,10444979.73,10468324.93,10500961.33,10518122.32,10532013.18,10542610.78,..,..
263,Agricultural land (sq. km),AG.LND.AGRI.K2,Sub-Saharan Africa (IDA & IBRD countries),TSS,9816756.063,9853189.038,9877442.818,9937208.871,9953818.379,10035523.13,...,10391986.8,10420918.54,10444995.23,10468340.43,10500976.83,10518137.82,10532028.68,10542626.28,..,..
264,Agricultural land (sq. km),AG.LND.AGRI.K2,Upper middle income,UMC,19045736.23,19029806.52,19040739.6,19052058.37,19066332.02,19074521.63,...,19025154.92,19028681.72,19016818.44,19010295.2,18996888.04,18980143.84,19017642.13,19023454.53,..,..


In [46]:
series_codes = wbd['Series Code'].unique()
series_names = wbd['Series Name'].unique()

In [47]:
titles = zip(series_names,series_codes)
print (set(titles))

{('Birth rate, crude (per 1,000 people)', 'SP.DYN.CBRT.IN'), ('Terrestrial and marine protected areas (% of total territorial area)', 'ER.PTD.TOTL.ZS'), ('Rural land area (sq. km)', 'AG.LND.TOTL.RU.K2'), ('CO2 emissions (kt)', 'EN.ATM.CO2E.KT'), ('Agricultural methane emissions (thousand metric tons of CO2 equivalent)', 'EN.ATM.METH.AG.KT.CE'), ('Population living in areas where elevation is below 5 meters (% of total population)', 'EN.POP.EL5M.ZS'), ('Urban land area where elevation is below 5 meters (% of total land area)', 'AG.LND.EL5M.UR.ZS'), ('Electricity production from hydroelectric sources (% of total)', 'EG.ELC.HYRO.ZS'), ('Population ages 15-64, female', 'SP.POP.1564.FE.IN'), ('Cereal production (metric tons)', 'AG.PRD.CREL.MT'), ('Electricity production from nuclear sources (% of total)', 'EG.ELC.NUCL.ZS'), ('Permanent cropland (% of land area)', 'AG.LND.CROP.ZS'), ('Community health workers (per 1,000 people)', 'SH.MED.CMHW.P3'), ('Nitrous oxide emissions in energy sector 

In [48]:
tbl_EN_ATM_CO2E_KT = wbd.loc[wbd['Series Code'] == "EN.ATM.CO2E.KT"]
tbl_AG_LND_TOTL_RU_K2 = wbd.loc[wbd['Series Code'] == "AG.LND.TOTL.RU.K2"]
tbl_EN_ATM_CO2E_KT

,Series Name,Series Code,Country Name,Country Code,2000 [YR2000],2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],...,2014 [YR2014],2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023]
5852,CO2 emissions (kt),EN.ATM.CO2E.KT,Afghanistan,AFG,1078.12,1088.638,1403.03,1653.207,1292.307,1961.711,...,9281.34,10057.59,9294.93,10022.78,10972.38,11238.83,8709.47,..,..,..
5853,CO2 emissions (kt),EN.ATM.CO2E.KT,Albania,ALB,3186.54,3234.2,3761.9,4137.4,4322.3,4095.9,...,5188,4797,4573.2,5403.7,5316.1,4993.3,4383.2,..,..,..
5854,CO2 emissions (kt),EN.ATM.CO2E.KT,Algeria,DZA,80046.8,78645.5,82404.3,88188.9,89492.7,94187.8,...,147735.2,156273,154654.3,157704.4,164534.1,170582.4,161563,..,..,..
5855,CO2 emissions (kt),EN.ATM.CO2E.KT,American Samoa,ASM,..,..,..,..,..,..,...,..,..,..,..,..,..,..,..,..,..
5856,CO2 emissions (kt),EN.ATM.CO2E.KT,Andorra,AND,523.952,523.952,531.28,534.944,560.592,575.248,...,461.664,465.328,468.992,465.328,494.64,479.984,448.884399,..,..,..
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6113,CO2 emissions (kt),EN.ATM.CO2E.KT,Sub-Saharan Africa,SSF,498620.9637,542940.9436,545144.7523,576823.2462,609493.5994,612882.4492,...,788024.602,770629.632,781618.462,791456.382,809339.8475,833713.85,760868.113,..,..,..
6114,CO2 emissions (kt),EN.ATM.CO2E.KT,Sub-Saharan Africa (excluding high income),SSA,498327.2637,542617.2436,544792.4523,576480.8462,609110.3994,612500.5492,...,787567.902,770129.832,781052.862,790885.682,808737.5475,833118.55,760269.413,..,..,..
6115,CO2 emissions (kt),EN.ATM.CO2E.KT,Sub-Saharan Africa (IDA & IBRD countries),TSS,498620.9637,542940.9436,545144.7523,576823.2462,609493.5994,612882.4492,...,788024.602,770629.632,781618.462,791456.382,809339.8475,833713.85,760868.113,..,..,..
6116,CO2 emissions (kt),EN.ATM.CO2E.KT,Upper middle income,UMC,7522848.982,7773270.878,8090933.049,8885518.073,9714080.938,10509932.36,...,15550983.46,15348154.71,15332252.1,15680729.34,16265198.32,16546520.02,16383995.51,..,..,..


## Task 2 - Data Analysis


Task 2. Data analysis (LO2, LO3)
Formulate data analysis questions which could be answered by analysing your chosen dataset. For each question provide graph/chart along with your own interpretation. Please find the sample data analysis questions formulated for the ONS census dataset.

### Task Questions

- Describe the relationship between type of land use and CO2 emmissions, by country.
- 

